# Influencer Advertising Effectiveness Analysis

This notebook provides a cleaned, reproducible analysis pipeline for the project **Creator Authenticity in Influencer Advertising**.

The goal is to test whether influencer ad performance is better explained by **brand alignment** or **creator authenticity**, especially visual consistency between a creator's organic content and sponsored content.

> Note: This notebook assumes the raw data files are stored in Google Drive under `/content/drive/MyDrive/rumor_data`. Large raw zip files are not included in the GitHub repository.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import ast
import json
import pickle
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from scipy.stats import ttest_ind
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

RAW_DIR = "/content/drive/MyDrive/rumor_data"
FIGURE_DIR = os.path.join(RAW_DIR, "github_figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

RANDOM_STATE = 42

## 2. Load and Parse Core Files

The project uses:

- `post_info.txt`: post index, username, sponsorship label, JSON filename, image filenames
- `profiles_influencers.zip`: creator profile metadata
- `profiles_brands.zip`: brand profile metadata
- `json_files-001.zip`: post metadata such as likes, comments, captions
- `img_fi*.zip`: post image archives

In [ ]:
post_info_path = os.path.join(RAW_DIR, "post_info.txt")

post_info = pd.read_csv(
    post_info_path,
    sep="	",
    header=None,
    names=["post_id", "user", "sponsored", "json_file", "image_file"]
)

# image_file is stored as a string representation of a list.
def parse_image_list(x):
    try:
        imgs = ast.literal_eval(str(x))
        return imgs if isinstance(imgs, list) else []
    except Exception:
        return []

post_info["image_list"] = post_info["image_file"].apply(parse_image_list)
post_info["first_image"] = post_info["image_list"].apply(lambda xs: xs[0] if len(xs) > 0 else None)

print("Total posts:", len(post_info))
post_info.head()

In [ ]:
# Parse influencer profiles.
influencer_zip = os.path.join(RAW_DIR, "profiles_influencers.zip")

profile_rows = []

with zipfile.ZipFile(influencer_zip, "r") as z:
    files = [name for name in z.namelist() if not name.endswith("/")]
    
    for file in files:
        try:
            username = os.path.basename(file)
            if username.endswith(".jpg"):
                continue
            
            with z.open(file) as f:
                content = f.read().decode("utf-8", errors="ignore").strip()
            
            parts = content.split("	")
            if len(parts) < 7:
                continue
            
            profile_rows.append({
                "user": username.lower().strip(),
                "profile_name": parts[0],
                "followers": int(parts[1]) if parts[1].isdigit() else np.nan,
                "followees": int(parts[2]) if parts[2].isdigit() else np.nan,
                "profile_posts": int(parts[3]) if parts[3].isdigit() else np.nan,
                "category": parts[6],
                "bio": parts[7] if len(parts) > 7 else ""
            })
        except Exception:
            continue

profile_df = pd.DataFrame(profile_rows)
profile_df["user"] = profile_df["user"].astype(str).str.lower().str.strip()

print("Influencer profiles:", profile_df.shape)
profile_df.head()

In [ ]:
# Parse brand profiles.
brand_zip = os.path.join(RAW_DIR, "profiles_brands.zip")

brand_rows = []

with zipfile.ZipFile(brand_zip, "r") as z:
    files = [name for name in z.namelist() if not name.endswith("/")]
    
    for file in files:
        try:
            brand_user = os.path.basename(file)
            if brand_user.endswith(".jpg"):
                continue
            
            with z.open(file) as f:
                content = f.read().decode("utf-8", errors="ignore").strip()
            
            parts = content.split("	")
            if len(parts) < 7:
                continue
            
            brand_rows.append({
                "brand_user": brand_user.lower().strip(),
                "brand_name": parts[0],
                "brand_followers": int(parts[1]) if parts[1].isdigit() else np.nan,
                "brand_category": parts[6],
                "brand_bio": parts[7] if len(parts) > 7 else ""
            })
        except Exception:
            continue

brand_df = pd.DataFrame(brand_rows)
print("Brand profiles:", brand_df.shape)
brand_df.head()

## 3. Image Matching and Balanced Sampling

We only keep posts whose images exist in the available image archives. Then, we filter creators who have at least 3 organic posts and 3 sponsored posts, and perform user-level balanced sampling.

In [ ]:
# Find image zip files.
image_zip_files = sorted([
    os.path.join(RAW_DIR, f)
    for f in os.listdir(RAW_DIR)
    if f.startswith("img_fi") and f.endswith(".zip")
])

print("Image zip files:", len(image_zip_files))
print(image_zip_files)

In [ ]:
# Build image lookup: image basename -> (zip index, inner path)
image_to_zipname = {}

for zip_index, zip_path in enumerate(image_zip_files):
    with zipfile.ZipFile(zip_path, "r") as z:
        for name in z.namelist():
            if name.lower().endswith((".jpg", ".jpeg", ".png")):
                image_to_zipname[os.path.basename(name)] = (zip_index, name)

print("Matched image filenames:", len(image_to_zipname))

In [ ]:
matched_posts = post_info[
    post_info["first_image"].isin(image_to_zipname.keys())
].copy()

matched_posts["user"] = matched_posts["user"].astype(str).str.lower().str.strip()

print("Matched posts:", len(matched_posts))
print(matched_posts["sponsored"].value_counts())

In [ ]:
# Keep creators with at least 3 organic and 3 sponsored posts.
user_counts = matched_posts.groupby("user")["sponsored"].value_counts().unstack(fill_value=0)

qualified_users = user_counts[
    (user_counts.get(0, 0) >= 3) &
    (user_counts.get(1, 0) >= 3)
].index

qualified_posts = matched_posts[matched_posts["user"].isin(qualified_users)].copy()

print("Qualified creators:", len(qualified_users))
print("Qualified posts:", len(qualified_posts))
print(qualified_posts["sponsored"].value_counts())

In [ ]:
# User-level balanced sampling.
balanced_list = []

for user, group in qualified_posts.groupby("user"):
    organic = group[group["sponsored"] == 0]
    sponsored = group[group["sponsored"] == 1]
    n = min(len(organic), len(sponsored))
    
    if n >= 3:
        balanced_list.append(organic.sample(n=n, random_state=RANDOM_STATE))
        balanced_list.append(sponsored.sample(n=n, random_state=RANDOM_STATE))

balanced_posts = pd.concat(balanced_list).reset_index(drop=True)

print("Balanced posts:", len(balanced_posts))
print(balanced_posts["sponsored"].value_counts())

In [ ]:
# Final CLIP sample.
clip_sample = balanced_posts.sample(
    n=min(8000, len(balanced_posts)),
    random_state=RANDOM_STATE
).reset_index(drop=True)

print("CLIP sample:", len(clip_sample))
print(clip_sample["sponsored"].value_counts())

## 4. Load JSON Metadata and Compute Engagement Rate

Likes and comments are extracted from Instagram JSON metadata. Engagement Rate is defined as:

\[
ER = rac{Likes + Comments}{Followers}
\]

In [ ]:
# Load only needed JSON metadata efficiently.
json_zip_files = sorted([
    os.path.join(RAW_DIR, f)
    for f in os.listdir(RAW_DIR)
    if f.startswith("json") and f.endswith(".zip")
])

needed_jsons = set(clip_sample["json_file"].dropna().unique())
meta_map = {}

for zip_path in json_zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        for name in tqdm(z.namelist(), desc=os.path.basename(zip_path)):
            base = os.path.basename(name)
            if base in needed_jsons:
                with z.open(name) as f:
                    meta_map[base] = json.load(f)

print("Needed JSON:", len(needed_jsons))
print("Loaded JSON:", len(meta_map))

In [ ]:
def extract_count(meta, candidates):
    if meta is None:
        return np.nan
    for key in candidates:
        if key in meta:
            value = meta[key]
            if isinstance(value, dict) and "count" in value:
                return value["count"]
            if isinstance(value, (int, float)):
                return value
    return np.nan

clip_sample["likes"] = clip_sample["json_file"].apply(
    lambda jf: extract_count(meta_map.get(jf), ["likes", "like_count", "edge_media_preview_like"])
)

clip_sample["comments"] = clip_sample["json_file"].apply(
    lambda jf: extract_count(meta_map.get(jf), ["comments", "comment_count", "edge_media_to_comment"])
)

print("Missing likes:", clip_sample["likes"].isna().sum())
print("Missing comments:", clip_sample["comments"].isna().sum())

In [ ]:
# Merge creator profile information.
clip_sample["user"] = clip_sample["user"].astype(str).str.lower().str.strip()

clip_sample = clip_sample.merge(
    profile_df[["user", "followers", "category", "bio"]],
    on="user",
    how="left"
)

clip_sample["engagement"] = clip_sample["likes"] + clip_sample["comments"]
clip_sample["ER"] = clip_sample["engagement"] / clip_sample["followers"]
clip_sample["log_followers"] = np.log1p(clip_sample["followers"])
clip_sample = clip_sample.replace([np.inf, -np.inf], np.nan)

print("Missing followers:", clip_sample["followers"].isna().sum())
print("Missing category:", clip_sample["category"].isna().sum())
clip_sample[["user", "sponsored", "likes", "comments", "followers", "ER", "category"]].head()

In [ ]:
# Optional: save intermediate sample.
clip_sample.to_pickle(os.path.join(RAW_DIR, "clip_sample_with_er_and_meta.pkl"))

with open(os.path.join(RAW_DIR, "image_to_zipname.pkl"), "wb") as f:
    pickle.dump(image_to_zipname, f)

print("Saved intermediate files.")

## 5. CLIP Image Embedding

If embeddings were already computed and saved, load them directly. Otherwise, set `RUN_CLIP = True` to compute embeddings from images.

> Computing 8,000 image embeddings can take several hours depending on the runtime.

In [ ]:
RUN_CLIP = False
EMBEDDING_PATH = os.path.join(RAW_DIR, "clip_embeddings_8000_expanded.npy")

In [ ]:
if RUN_CLIP:
    import torch
    from PIL import Image
    from transformers import CLIPProcessor, CLIPModel
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)
    
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    clip_model.eval()
    
    def load_image_from_zip(image_name):
        zip_index, inner_path = image_to_zipname[image_name]
        zip_path = image_zip_files[zip_index]
        with zipfile.ZipFile(zip_path, "r") as z:
            with z.open(inner_path) as f:
                return Image.open(f).convert("RGB")
    
    batch_size = 64
    embeddings = []
    image_names = clip_sample["first_image"].tolist()
    
    for i in tqdm(range(0, len(image_names), batch_size)):
        batch_names = image_names[i:i+batch_size]
        images = [load_image_from_zip(name) for name in batch_names]
        
        inputs = clip_processor(images=images, return_tensors="pt", padding=True).to(device)
        
        with torch.no_grad():
            out = clip_model.vision_model(pixel_values=inputs["pixel_values"])
            pooled = out.pooler_output
            feats = clip_model.visual_projection(pooled)
            feats = feats / feats.norm(dim=1, keepdim=True)
        
        embeddings.append(feats.cpu().numpy())
    
    clip_embeddings = np.vstack(embeddings)
    np.save(EMBEDDING_PATH, clip_embeddings)
else:
    clip_embeddings = np.load(EMBEDDING_PATH)

print("Embeddings:", clip_embeddings.shape)

## 6. Style Fit and Visual Novelty

For each creator, we compute an organic visual baseline from their organic posts. Then, each post is compared to the creator's baseline using cosine similarity.

\[
StyleFit = \cos(v_{organic}, v_{post})
\]

\[
Novelty = 1 - StyleFit
\]

In [ ]:
embedding_df = clip_sample.copy()
embedding_df["embedding"] = list(clip_embeddings)

# Organic visual baseline per creator.
organic_df = embedding_df[embedding_df["sponsored"] == 0].copy()
creator_baselines = {}

for user, group in organic_df.groupby("user"):
    vecs = np.stack(group["embedding"].values)
    mean_vec = vecs.mean(axis=0)
    mean_vec = mean_vec / np.linalg.norm(mean_vec)
    creator_baselines[user] = mean_vec

print("Creator baselines:", len(creator_baselines))

In [ ]:
def cosine_to_baseline(row):
    user = row["user"]
    if user not in creator_baselines:
        return np.nan
    baseline = creator_baselines[user]
    current = row["embedding"]
    return float(np.dot(baseline, current))

embedding_df["style_fit"] = embedding_df.apply(cosine_to_baseline, axis=1)
embedding_df["novelty"] = 1 - embedding_df["style_fit"]

ad_df = embedding_df[embedding_df["sponsored"] == 1].dropna(
    subset=["style_fit", "novelty", "ER", "log_followers"]
).copy()

print("Usable ad samples:", len(ad_df))
embedding_df[["sponsored", "style_fit", "novelty", "ER"]].describe()

## 7. Main Experiments

In [ ]:
# 7.1 Sponsored vs Organic ER
summary_er = embedding_df.groupby("sponsored")["ER"].agg(["count", "mean", "median", "std"])
print(summary_er)

organic_er = embedding_df[embedding_df["sponsored"] == 0]["ER"].dropna()
sponsored_er = embedding_df[embedding_df["sponsored"] == 1]["ER"].dropna()

t, p = ttest_ind(organic_er, sponsored_er, equal_var=False)
print("t =", t)
print("p =", p)

In [ ]:
# 7.2 Organic vs Sponsored Style Fit
style_summary = embedding_df.groupby("sponsored")["style_fit"].agg(["count", "mean", "median", "std"])
print(style_summary)

organic_style = embedding_df[embedding_df["sponsored"] == 0]["style_fit"].dropna()
sponsored_style = embedding_df[embedding_df["sponsored"] == 1]["style_fit"].dropna()

t, p = ttest_ind(organic_style, sponsored_style, equal_var=False)
print("t =", t)
print("p =", p)

In [ ]:
# 7.3 Low vs High Novelty Engagement
threshold = ad_df["novelty"].median()
ad_df["novelty_group"] = np.where(ad_df["novelty"] > threshold, "High Novelty", "Low Novelty")

novelty_group_result = ad_df.groupby("novelty_group")["ER"].agg(["count", "mean", "median", "std"])
print(novelty_group_result)

In [ ]:
# 7.4 Main OLS Regression
model = smf.ols(
    "ER ~ novelty + log_followers",
    data=ad_df
).fit()

print(model.summary())

In [ ]:
# 7.5 Robustness Check: log-transformed ER
ad_df["log_ER"] = np.log1p(ad_df["ER"])

model_log = smf.ols(
    "log_ER ~ novelty + log_followers",
    data=ad_df
).fit()

print(model_log.summary())

In [ ]:
# 7.6 Robustness Check: remove top 1% ER outliers
cutoff = ad_df["ER"].quantile(0.99)
robust_df = ad_df[ad_df["ER"] <= cutoff].copy()

model_outlier = smf.ols(
    "ER ~ novelty + log_followers",
    data=robust_df
).fit()

print("Original ad samples:", len(ad_df))
print("After removing top 1% ER outliers:", len(robust_df))
print(model_outlier.summary())

## 8. Category and Semantic Fit Checks

These checks test whether brand alignment explains engagement. In the final analysis, these features were weaker or unstable compared to visual authenticity.

In [ ]:
def extract_caption(meta):
    if meta is None:
        return None
    edges = meta.get("edge_media_to_caption", {}).get("edges", [])
    if len(edges) == 0:
        return None
    return edges[0].get("node", {}).get("text")

def extract_brand_user(meta):
    if meta is None:
        return None
    
    sponsor = meta.get("edge_media_to_sponsor_user", {})
    edges = sponsor.get("edges", []) if isinstance(sponsor, dict) else []
    if len(edges) > 0:
        node = edges[0].get("node", {})
        user = node.get("sponsor", node.get("user", node))
        if isinstance(user, dict):
            return user.get("username")
    
    tagged = meta.get("edge_media_to_tagged_user", {})
    edges = tagged.get("edges", []) if isinstance(tagged, dict) else []
    if len(edges) > 0:
        node = edges[0].get("node", {})
        user = node.get("user", {})
        if isinstance(user, dict):
            return user.get("username")
    
    return None

ad_df["brand_user"] = ad_df["json_file"].apply(lambda jf: extract_brand_user(meta_map.get(jf)))
ad_df["caption"] = ad_df["json_file"].apply(lambda jf: extract_caption(meta_map.get(jf)))
ad_df["brand_user"] = ad_df["brand_user"].astype(str).str.lower().str.strip().replace("none", np.nan)

print("Brand user detected:", ad_df["brand_user"].notna().sum())

In [ ]:
fit_df = ad_df.merge(brand_df, on="brand_user", how="left")
fit_only_df = fit_df[fit_df["brand_category"].notna()].copy()
fit_only_df["category_fit"] = fit_only_df["category"] == fit_only_df["brand_category"]

print("Matched brand category:", len(fit_only_df))
print(fit_only_df["category_fit"].value_counts())

if fit_only_df["category_fit"].nunique() == 2:
    print(fit_only_df.groupby("category_fit")["ER"].agg(["count", "mean", "median", "std"]))
    fit_er = fit_only_df[fit_only_df["category_fit"]]["ER"]
    nonfit_er = fit_only_df[~fit_only_df["category_fit"]]["ER"]
    t, p = ttest_ind(fit_er, nonfit_er, equal_var=False)
    print("t =", t)
    print("p =", p)

In [ ]:
# Optional semantic fit analysis using sentence-transformers.
RUN_SEMANTIC = False

if RUN_SEMANTIC:
    !pip -q install sentence-transformers
    from sentence_transformers import SentenceTransformer
    
    semantic_df = fit_df.dropna(subset=["caption", "brand_bio", "ER"]).copy()
    print("Semantic samples:", len(semantic_df))
    
    text_model = SentenceTransformer("all-MiniLM-L6-v2")
    caption_emb = text_model.encode(semantic_df["caption"].tolist(), show_progress_bar=True, batch_size=64)
    brand_emb = text_model.encode(semantic_df["brand_bio"].tolist(), show_progress_bar=True, batch_size=64)
    
    # SentenceTransformer embeddings are normalized enough for dot-product similarity in this simplified check.
    semantic_df["semantic_fit"] = np.sum(caption_emb * brand_emb, axis=1)
    threshold = semantic_df["semantic_fit"].median()
    semantic_df["semantic_group"] = semantic_df["semantic_fit"] > threshold
    
    print(semantic_df["semantic_fit"].describe())
    print(semantic_df.groupby("semantic_group")["ER"].agg(["count", "mean", "median", "std"]))
    
    high = semantic_df[semantic_df["semantic_group"]]["ER"]
    low = semantic_df[~semantic_df["semantic_group"]]["ER"]
    t, p = ttest_ind(high, low, equal_var=False)
    print("t =", t)
    print("p =", p)

## 9. Visualization

The following figures are used in the project paper and GitHub README.

In [ ]:
# Figure 1: Style Fit Comparison
plot_df = embedding_df.dropna(subset=["style_fit"]).copy()
plot_df["type"] = plot_df["sponsored"].map({0: "Organic", 1: "Sponsored"})

plt.figure(figsize=(6, 5))
plot_df.boxplot(column="style_fit", by="type")
plt.title("Style Fit: Organic vs Sponsored")
plt.suptitle("")
plt.xlabel("")
plt.ylabel("Style Fit")
plt.savefig(os.path.join(FIGURE_DIR, "figure1_style_fit.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 2: Novelty vs Engagement Scatter
plt.figure(figsize=(6, 5))
plt.scatter(ad_df["novelty"], ad_df["ER"], alpha=0.4)
plt.xlabel("Novelty")
plt.ylabel("Engagement Rate")
plt.title("Visual Novelty vs Engagement")
plt.savefig(os.path.join(FIGURE_DIR, "figure2_novelty_scatter.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 3: Low vs High Novelty Bar Chart
novelty_bar = ad_df.groupby("novelty_group")["ER"].mean().reindex(["Low Novelty", "High Novelty"])

plt.figure(figsize=(6, 5))
plt.bar(novelty_bar.index, novelty_bar.values)
plt.title("Engagement Rate by Novelty Group")
plt.ylabel("Mean Engagement Rate")
plt.savefig(os.path.join(FIGURE_DIR, "figure3_novelty_bar.png"), dpi=300, bbox_inches="tight")
plt.show()

print(novelty_bar)

## 10. Save Final Analysis Data

In [ ]:
embedding_df.to_pickle(os.path.join(RAW_DIR, "final_analysis_df.pkl"))
ad_df.to_pickle(os.path.join(RAW_DIR, "ad_df.pkl"))

print("Saved final analysis files.")

## Final Takeaway

The main empirical result is that **visual novelty negatively predicts engagement** in sponsored posts. Ads that deviate from a creator's established visual identity tend to perform worse, supporting the claim:

> Consumers respond more strongly to creator authenticity than brand authenticity.